# Notebook 03 — Retrieve Better: Bi-Encoder Over-Fetch + Cross-Encoder Rerank

## Tujuan Pembelajaran

Retrieval satu tahap dengan **dense / bi-encoder** bersifat cepat tapi kasar — bagus untuk *recall* (dokumen relevan kemungkinan besar ADA di hasil), tapi lemah untuk *precision* di posisi teratas (urutan belum tentu benar).

**Solusi dua tahap:**
1. **Over-fetch** — bi-encoder mengambil banyak kandidat (mis. 12) dengan biaya murah.
2. **Rerank** — cross-encoder membaca pasangan (query, passage) BERSAMA dan mengurutkan ulang kandidat tersebut dengan akurasi yang jauh lebih tinggi, lalu kita simpan hanya top-3 sebagai konteks generator.

Notebook ini mencakup:
- Membangun FAISS index dengan bi-encoder multilingual
- Over-fetch kandidat
- Reranking dengan `BAAI/bge-reranker-v2-m3`
- Visualisasi rank-change
- Grounded answer generation dengan Qwen2.5-3B-Instruct (4-bit)

In [ ]:
!pip install -q "transformers<5" sentence-transformers faiss-cpu accelerate bitsandbytes

In [ ]:
import os, sys
REPO = "navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/chmdznr/navasena-gen-ml-course.git
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))
from tools.rag_utils import rank_change_table

## Bi-Encoder vs Cross-Encoder

| Aspek | Bi-Encoder | Cross-Encoder |
|-------|-----------|---------------|
| **Cara kerja** | Embed query & dokumen **TERPISAH** | Proses pasangan (query, passage) **BERSAMA** |
| **Kecepatan** | Cepat — vektor bisa pra-hitung offline | Lambat — setiap pasangan butuh satu forward pass |
| **Akurasi ranking** | Kasar — tak melihat interaksi kata per kata | Tajam — memodelkan interaksi query↔passage secara penuh |
| **Skalabilitas** | Jutaan dokumen (ANN search) | Puluhan–ratusan kandidat |

**Strategi dua tahap:**
- **Tahap 1 (bi-encoder)** — saring dari jutaan → puluhan kandidat (over-fetch)
- **Tahap 2 (cross-encoder)** — urutkan ulang puluhan kandidat → simpan top-K

Bi-encoder menjamin *recall*; cross-encoder menjamin *precision* di posisi teratas.

In [ ]:
# Korpus sengaja dibuat dengan jebakan leksikal: banyak kalimat memuat kata
# "Komodo" (Pulau/Taman Komodo) yang MIRIP secara kata tapi BUKAN jawaban untuk
# pertanyaan tentang hewannya. Inilah yang membuat reranking terlihat bekerja.
corpus = [
    "Komodo adalah spesies kadal terbesar di dunia dan merupakan hewan endemik Indonesia.",
    "Habitat alami komodo berada di Nusa Tenggara Timur, terutama di beberapa pulau kecil.",
    "Taman Nasional Komodo ditetapkan sebagai Situs Warisan Dunia UNESCO pada tahun 1991.",
    "Pulau Komodo adalah destinasi wisata populer untuk melihat satwa langka dari dekat.",
    "Wisatawan yang berkunjung ke Pulau Komodo biasanya juga singgah ke Pulau Padar.",
    "Festival budaya di Pulau Komodo menampilkan tarian tradisional masyarakat setempat.",
    "Tiket masuk Taman Nasional Komodo berbeda untuk wisatawan domestik dan mancanegara.",
    "Komodo dapat berlari cukup cepat dalam jarak pendek untuk memburu mangsanya.",
    "Air liur komodo mengandung bakteri dan bisa yang dapat melumpuhkan mangsa.",
    "Candi Borobudur adalah candi Buddha terbesar di dunia, terletak di Magelang, Jawa Tengah.",
    "Candi Prambanan adalah kompleks candi Hindu terbesar di Indonesia.",
    "Gunung Bromo terkenal dengan pemandangan matahari terbit yang menakjubkan.",
    "Kawah Gunung Bromo masih aktif dan mengeluarkan asap belerang.",
    "Raja Ampat di Papua Barat memiliki keanekaragaman hayati laut tertinggi di dunia.",
    "Pulau Bali dikenal sebagai Pulau Dewata karena banyaknya pura.",
    "Danau Toba adalah danau vulkanik terbesar di Asia Tenggara, terbentuk dari letusan supervolcano.",
    "Orangutan adalah kera besar yang hidup di hutan Kalimantan dan Sumatra.",
    "Harimau Sumatra adalah subspesies harimau yang terancam punah.",
    "Bunga Rafflesia arnoldii adalah bunga terbesar di dunia dan tumbuh di Sumatra.",
    "Burung cendrawasih sering disebut burung surga dan berasal dari Papua.",
    "Nasi goreng adalah salah satu makanan khas Indonesia yang populer di dunia.",
    "Batik Indonesia diakui UNESCO sebagai Warisan Budaya Takbenda.",
    "Indonesia memiliki lebih dari 17.000 pulau yang tersebar dari Sabang hingga Merauke.",
    "Gajah Sumatra adalah mamalia darat terbesar di Indonesia.",
]
print(f"{len(corpus)} passage dalam korpus")

In [ ]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
corpus_vecs = embedder.encode(corpus, convert_to_numpy=True).astype("float32")
dim = corpus_vecs.shape[1]
index = faiss.IndexFlatL2(dim)   # cosine/IndexFlatIP diperkenalkan di nb05; L2 cukup untuk tahap penyaringan
index.add(corpus_vecs)
print("dim:", dim, "| index size:", index.ntotal)

## Tahap 1: Over-Fetch dengan Bi-Encoder

Ambil lebih banyak kandidat dari yang dibutuhkan (K=12) supaya jawaban benar kemungkinan besar **ADA** di kandidat walau belum di posisi #1.

Bi-encoder murah karena:
- Vektor dokumen sudah pra-hitung di index
- Hanya query yang di-embed saat runtime
- FAISS menggunakan ANN (approximate nearest neighbor) yang sangat cepat

Jadi over-fetch dengan K=12 hampir tidak lebih mahal dari K=3.

In [ ]:
query = "Hewan kadal terbesar di dunia hidup di mana?"
K = 12
qv = embedder.encode([query], convert_to_numpy=True).astype("float32")
dist, idx = index.search(qv, K)
candidate_ids = idx[0].tolist()
bi_scores = [float(-d) for d in dist[0]]   # negate L2 distance so higher = lebih mirip
print(f"Query: {query}\n\nTahap 1 — bi-encoder over-fetch (top {K}):")
for rank, cid in enumerate(candidate_ids, 1):
    print(f"  bi#{rank:2d} | doc {cid:2d} | {corpus[cid][:72]}")

## Tahap 2: Rerank dengan Cross-Encoder

Cross-encoder membaca pasangan **(query, passage) BERSAMA** dalam satu forward pass model, sehingga bisa memodelkan interaksi semantik yang detail — misalnya kata "hidup di mana" berinteraksi dengan "berada di Nusa Tenggara Timur" vs "destinasi wisata".

Hasilnya adalah skor relevansi yang jauh lebih tajam. Kita simpan **top-3** sebagai konteks untuk generator.

Model: `BAAI/bge-reranker-v2-m3` — cross-encoder multilingual, non-gated, muat di T4.

In [ ]:
from sentence_transformers import CrossEncoder
# bge-reranker-v2-m3: cross-encoder multilingual, non-gated, muat di T4.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
pairs = [(query, corpus[cid]) for cid in candidate_ids]
rerank_scores = reranker.predict(pairs).tolist()
rows = rank_change_table(candidate_ids, bi_scores, rerank_scores, top_k=3)
top3 = [r for r in rows if r["kept"]]
print("Tahap 2 — setelah cross-encoder rerank (top 3 disimpan):")
for r in top3:
    print(f"  rerank#{r['rerank_rank']} | doc {r['doc_id']} (asalnya bi#{r['bi_rank']}, delta {r['delta']:+d}) | {corpus[r['doc_id']][:60]}")

## Membaca Rank-Change

Kolom **delta** = bi_rank − rerank_rank:
- **delta > 0** (hijau) — dokumen **NAIK** setelah rerank: cross-encoder menilainya lebih relevan dari perkiraan bi-encoder
- **delta < 0** (merah) — dokumen **TURUN**: bi-encoder terlalu optimis karena kesamaan leksikal saja
- **delta = 0** (abu) — urutan tidak berubah

Harapan kita: passage "Komodo adalah spesies kadal terbesar di dunia..." (doc 0) dipromosikan ke atas, sedangkan passage tentang Pulau/Taman Komodo yang hanya mirip secara kata turun peringkat — inilah bukti bahwa cross-encoder memahami *makna*, bukan sekadar overlap token.

In [ ]:
import matplotlib.pyplot as plt
print(f"{'doc':>3} {'bi':>3} {'rr':>3} {'delta':>6} {'kept':>5}  teks")
for r in rows:
    print(f"{r['doc_id']:>3} {r['bi_rank']:>3} {r['rerank_rank']:>3} {r['delta']:>+6} {str(r['kept']):>5}  {corpus[r['doc_id']][:48]}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter([r['bi_score'] for r in rows], [r['rerank_score'] for r in rows])
for r in rows:
    ax[0].annotate(str(r['doc_id']), (r['bi_score'], r['rerank_score']), fontsize=8)
ax[0].set_xlabel('skor bi-encoder'); ax[0].set_ylabel('skor cross-encoder')
ax[0].set_title('Korelasi lemah = reranker menambah sinyal baru')

docs = [str(r['doc_id']) for r in rows]; deltas = [r['delta'] for r in rows]
ax[1].bar(docs, deltas, color=['green' if d > 0 else ('red' if d < 0 else 'grey') for d in deltas])
ax[1].axhline(0, color='black', linewidth=0.6)
ax[1].set_xlabel('doc id'); ax[1].set_ylabel('perubahan rank (naik = +)')
ax[1].set_title('Reranker menggeser urutan')
plt.tight_layout(); plt.show()

## Biaya: Mengapa Cross-Encoder Tidak Dipakai untuk Seluruh Korpus?

Cross-encoder **mahal** karena setiap pasangan (query, passage) membutuhkan satu forward pass model penuh — tidak ada vektor yang bisa pra-hitung atau diindeks.

- Korpus 1 juta dokumen × 1 forward pass/pasangan = **1 juta inference** per query
- Dengan bi-encoder FAISS: index search ~1ms; hanya **K forward pass** untuk reranker

Itulah sebabnya arsitektur dua tahap ini menjadi standar industri: bi-encoder menjaga agar set kandidat tetap kecil, sehingga cross-encoder tetap layak secara komputasi.

In [ ]:
import time
sizes = [s for s in [5, 10, 20, 40] if s <= len(corpus)]
d_all, i_all = index.search(qv, min(max(sizes), len(corpus)))
pool = i_all[0].tolist()
t1, t2 = [], []
for n in sizes:
    s0 = time.perf_counter(); _ = embedder.encode([query], convert_to_numpy=True); _ = index.search(qv, n); t1.append(time.perf_counter() - s0)
    pr = [(query, corpus[c]) for c in pool[:n]]
    s0 = time.perf_counter(); _ = reranker.predict(pr); t2.append(time.perf_counter() - s0)
plt.plot(sizes, t1, marker='o', label='Tahap 1 (embed + FAISS)')
plt.plot(sizes, t2, marker='o', label='Tahap 2 (cross-encoder rerank)')
plt.xlabel('jumlah kandidat'); plt.ylabel('detik'); plt.legend()
plt.title('Tahap 2 tumbuh ~linear dengan jumlah kandidat, Tahap 1 relatif datar'); plt.show()

## Grounded Answer Generation

Sekarang kita tutup loop RAG: 3 passage ter-rerank dikirim sebagai **konteks** ke model generator.

Generator diminta untuk:
1. Menjawab **HANYA** berdasarkan konteks yang diberikan (grounded)
2. Menyertakan **sitasi** `(doc N)` sehingga jawaban bisa diverifikasi

Model: `Qwen/Qwen2.5-3B-Instruct` dengan 4-bit quantization (NF4) — muat di T4 GPU Colab.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model_name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)
gen = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")

context = "\n".join(f"[doc {r['doc_id']}] {corpus[r['doc_id']]}" for r in top3)
messages = [
    {"role": "system", "content": "Jawab HANYA berdasarkan konteks. Sertakan id dokumen sebagai sitasi, mis. (doc 0)."},
    {"role": "user", "content": f"Konteks:\n{context}\n\nPertanyaan: {query}"},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to(gen.device)
out = gen.generate(**inputs, max_new_tokens=160, do_sample=False)
print(tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip())

## Ringkasan & Jembatan ke Notebook 04

**Yang sudah kita pelajari di nb03:**

| Tahap | Komponen | Peran |
|-------|----------|-------|
| 1 | Bi-encoder + FAISS | Over-fetch cepat → jamin *recall* |
| 2 | Cross-encoder (bge-reranker-v2-m3) | Rerank → tingkatkan *precision* posisi teratas |
| 3 | Generator (Qwen2.5-3B-Instruct) | Grounded answer dengan sitasi |

**Reranking meningkatkan *context precision*** — passage yang benar-benar menjawab pertanyaan naik ke posisi teratas, bukan hanya yang mengandung kata yang sama.

**Di Notebook 04 (Measure / RAGAS)**, kita akan **mengukur efek ini secara kuantitatif**: dengan query yang sama, bandingkan pipeline tanpa rerank vs dengan rerank menggunakan metrik seperti `context_precision`, `context_recall`, dan `answer_relevancy`. Angka akan membuktikan apa yang baru saja kita lihat secara visual.